# QUANT — Leave-One-Partition-Out Class-Weight and Threshold Search

This notebook trains the five-feature QUANT model using **leave-one-partition-out (LOPO) validation** instead of a random validation split.

## Data design

| Role | SWAN-SF partitions |
|---|---|
| Model development | 1, 2, 3, and 5 |
| Final held-out test | 4 |

The four validation folds are:

1. train on 2, 3, 5 → validate on 1
2. train on 1, 3, 5 → validate on 2
3. train on 1, 2, 5 → validate on 3
4. train on 1, 2, 3 → validate on 5

For every class-weight candidate, the notebook generates out-of-fold probabilities for all development samples. Probability thresholds are evaluated separately in each held-out partition. The winning class weight and threshold are selected by **mean validation TSS across the four folds**, with HSS and pooled out-of-fold TSS used as tie-breakers.

Partition 4 is not used for model or threshold selection. It is evaluated only after the winning configuration is frozen.

## Five input channels

1. `TOTUSJH`
2. `TOTBSQ`
3. `TOTPOT`
4. `TOTUSJZ`
5. `ABSNJZH`

Each input sample is expected to contain 5 channels and 60 time steps.


## 1. Mount Google Drive and install packages


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import importlib.metadata
import subprocess
import sys

REQUIRED_AEON_VERSION = "1.5.0"

try:
    installed_aeon = importlib.metadata.version("aeon")
except importlib.metadata.PackageNotFoundError:
    installed_aeon = None

if installed_aeon != REQUIRED_AEON_VERSION:
    print(
        f"Installing aeon=={REQUIRED_AEON_VERSION} "
        f"(currently installed: {installed_aeon})"
    )
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"aeon=={REQUIRED_AEON_VERSION}",
        "scikit-learn",
        "joblib",
        "pandas",
        "tqdm",
    ])
else:
    print(f"aeon=={REQUIRED_AEON_VERSION} is already installed.")


## 2. Imports and configuration


In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import time

import aeon
import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from tqdm.auto import tqdm

from aeon.transformations.collection.interval_based import QUANTTransformer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

print("aeon version:", aeon.__version__)
print("scikit-learn version:", sklearn.__version__)


In [ ]:
# =============================
# Data and output paths
# =============================
DATA_DIR = Path(
    "/content/drive/MyDrive/solar_flare_forecasting/Data/5_features_standardized"
)
OUTPUT_DIR = DATA_DIR / "quant_5_feature_lopo_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =============================
# Expected dataset structure
# =============================
FEATURES_TO_USE = [
    "TOTUSJH",
    "TOTBSQ",
    "TOTPOT",
    "TOTUSJZ",
    "ABSNJZH",
]
DEVELOPMENT_PARTITIONS = [1, 2, 3, 5]
FINAL_TEST_PARTITION = 4

# =============================
# Class-weight and threshold grid
# =============================
CLASS_WEIGHT_OPTIONS = [
    ("none", None),
    ("balanced", "balanced"),
    ("positive_5x", {0: 1, 1: 5}),
    ("positive_10x", {0: 1, 1: 10}),
]
THRESHOLDS = np.linspace(0.05, 0.95, 19)
DEFAULT_THRESHOLD = 0.50

# =============================
# QUANT and Extra Trees settings
# Retained from the supplied notebook.
# =============================
QUANT_INTERVAL_DEPTH = 6
QUANT_QUANTILE_DIVISOR = 4
N_ESTIMATORS = 200
RANDOM_SEED = 42
PREDICT_BATCH_SIZE = 20_000

# =============================
# Validation and caching
# =============================
DROP_LOW_VARIANCE_CASES = True
VARIANCE_THRESHOLD = 1e-7
VALIDATION_BATCH_SIZE = 50_000
CACHE_TRANSFORMED_FEATURES = True
REUSE_VALID_CACHE = True
SAVE_FINAL_MODEL = True

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Development partitions:", DEVELOPMENT_PARTITIONS)
print("Final test partition:", FINAL_TEST_PARTITION)
print("Features:", FEATURES_TO_USE)
print("Class weights:", [name for name, _ in CLASS_WEIGHT_OPTIONS])
print("Thresholds:", THRESHOLDS)


## 3. Locate and load the standardized tensors and partition sidecars


In [ ]:
def require_file(data_dir: Path, filename: str) -> Path:
    path = data_dir / filename
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")
    return path


x_train_path = require_file(DATA_DIR, "X_train.npy")
x_test_path = require_file(DATA_DIR, "X_test.npy")
y_train_path = require_file(DATA_DIR, "y_train.npy")
y_test_path = require_file(DATA_DIR, "y_test.npy")
train_partition_path = require_file(DATA_DIR, "train_partition_id.npy")
test_partition_path = require_file(DATA_DIR, "test_partition_id.npy")
feature_names_path = require_file(DATA_DIR, "selected_feature_names.json")

paths = {
    "X_train": x_train_path,
    "X_test": x_test_path,
    "y_train": y_train_path,
    "y_test": y_test_path,
    "train_partition_id": train_partition_path,
    "test_partition_id": test_partition_path,
    "selected_feature_names": feature_names_path,
}

for name, path in paths.items():
    print(f"{name:24s} {path}")


In [ ]:
# Memory-map the feature tensors so loading does not immediately duplicate them in RAM.
X_train_raw = np.load(x_train_path, mmap_mode="r")
X_test_raw = np.load(x_test_path, mmap_mode="r")
y_train_raw = np.load(y_train_path)
y_test_raw = np.load(y_test_path)
train_partition_raw = np.load(train_partition_path)
test_partition_raw = np.load(test_partition_path)

with open(feature_names_path, "r") as f:
    selected_feature_names = json.load(f)

print("Raw shapes and dtypes")
print("X_train:            ", X_train_raw.shape, X_train_raw.dtype)
print("X_test:             ", X_test_raw.shape, X_test_raw.dtype)
print("y_train:            ", y_train_raw.shape, y_train_raw.dtype)
print("y_test:             ", y_test_raw.shape, y_test_raw.dtype)
print("train_partition_id: ", train_partition_raw.shape, train_partition_raw.dtype)
print("test_partition_id:  ", test_partition_raw.shape, test_partition_raw.dtype)
print("Feature names:      ", selected_feature_names)


## 4. Validate tensor layout, alignment, feature order, and partition assignments


In [ ]:
def ensure_aeon_layout(X, feature_count):
    """Return a view in aeon layout: cases x channels x timepoints."""
    if X.ndim != 3:
        raise ValueError(f"Expected a 3D tensor, received shape {X.shape}")

    if X.shape[1] == feature_count:
        return X, "cases x channels x timepoints"
    if X.shape[2] == feature_count:
        return X.transpose(0, 2, 1), "cases x timepoints x channels -> transposed"

    raise ValueError(
        f"Could not locate the feature axis in tensor shape {X.shape}; "
        f"expected {feature_count} feature channels."
    )


if selected_feature_names != FEATURES_TO_USE:
    raise ValueError(
        "selected_feature_names.json does not match the required channel order.\n"
        f"Expected: {FEATURES_TO_USE}\n"
        f"Found:    {selected_feature_names}"
    )

X_train_aeon, train_layout = ensure_aeon_layout(
    X_train_raw, len(selected_feature_names)
)
X_test_aeon, test_layout = ensure_aeon_layout(
    X_test_raw, len(selected_feature_names)
)

if X_train_aeon.shape[0] != len(y_train_raw):
    raise ValueError("X_train and y_train are not aligned.")
if X_train_aeon.shape[0] != len(train_partition_raw):
    raise ValueError("X_train and train_partition_id are not aligned.")
if X_test_aeon.shape[0] != len(y_test_raw):
    raise ValueError("X_test and y_test are not aligned.")
if X_test_aeon.shape[0] != len(test_partition_raw):
    raise ValueError("X_test and test_partition_id are not aligned.")

observed_development_partitions = sorted(
    int(x) for x in np.unique(train_partition_raw)
)
observed_test_partitions = sorted(int(x) for x in np.unique(test_partition_raw))

if observed_development_partitions != DEVELOPMENT_PARTITIONS:
    raise ValueError(
        "Unexpected training partitions. "
        f"Expected {DEVELOPMENT_PARTITIONS}, found {observed_development_partitions}."
    )
if observed_test_partitions != [FINAL_TEST_PARTITION]:
    raise ValueError(
        "The test split must contain only partition 4. "
        f"Found {observed_test_partitions}."
    )

print("Training layout:", train_layout)
print("Test layout:    ", test_layout)
print("Aeon train shape:", X_train_aeon.shape)
print("Aeon test shape: ", X_test_aeon.shape)
print("Training partitions:", observed_development_partitions)
print("Test partitions:    ", observed_test_partitions)


## 5. Check class distributions and remove invalid low-variance cases


In [ ]:
def class_count_table(y):
    values, counts = np.unique(y, return_counts=True)
    table = pd.DataFrame({"class": values, "count": counts})
    table["percent"] = 100 * table["count"] / len(y)
    return table


def partition_class_table(partition_ids, y):
    frame = pd.DataFrame({"partition": partition_ids, "label": y})
    counts = pd.crosstab(frame["partition"], frame["label"])
    counts.columns = [f"class_{column}" for column in counts.columns]
    counts["total"] = counts.sum(axis=1)
    for column in [c for c in counts.columns if c.startswith("class_")]:
        counts[f"{column}_percent"] = 100 * counts[column] / counts["total"]
    return counts.reset_index()


def case_validity_mask(X, variance_threshold, batch_size):
    finite_mask = np.ones(X.shape[0], dtype=bool)
    variance_mask = np.ones(X.shape[0], dtype=bool)

    for start in tqdm(
        range(0, X.shape[0], batch_size),
        desc="Checking finite values and per-channel variance",
    ):
        stop = min(start + batch_size, X.shape[0])
        chunk = np.asarray(X[start:stop])
        finite_mask[start:stop] = np.isfinite(chunk).all(axis=(1, 2))
        channel_std = np.std(chunk, axis=2)
        variance_mask[start:stop] = (channel_std > variance_threshold).all(axis=1)

    return finite_mask, variance_mask


def apply_case_mask(X, y, partition_ids, keep_mask, split_name):
    removed = int((~keep_mask).sum())
    print(f"{split_name}: retaining {keep_mask.sum():,} / {len(keep_mask):,} cases")
    print(f"{split_name}: removing {removed:,} cases")

    original_indices = np.flatnonzero(keep_mask)
    if removed == 0:
        return X, y, partition_ids, original_indices

    return (
        np.asarray(X[keep_mask], dtype=np.float32),
        y[keep_mask],
        partition_ids[keep_mask],
        original_indices,
    )


train_finite, train_variance = case_validity_mask(
    X_train_aeon, VARIANCE_THRESHOLD, VALIDATION_BATCH_SIZE
)
test_finite, test_variance = case_validity_mask(
    X_test_aeon, VARIANCE_THRESHOLD, VALIDATION_BATCH_SIZE
)

if not train_finite.all():
    raise ValueError(
        f"Training tensor contains {(~train_finite).sum():,} cases with NaN or infinite values."
    )
if not test_finite.all():
    raise ValueError(
        f"Test tensor contains {(~test_finite).sum():,} cases with NaN or infinite values."
    )

train_keep = train_finite & (train_variance if DROP_LOW_VARIANCE_CASES else True)
test_keep = test_finite & (test_variance if DROP_LOW_VARIANCE_CASES else True)

X_train, y_train, train_partition_id, train_original_indices = apply_case_mask(
    X_train_aeon,
    y_train_raw,
    train_partition_raw,
    train_keep,
    "Training data",
)
X_test, y_test, test_partition_id, test_original_indices = apply_case_mask(
    X_test_aeon,
    y_test_raw,
    test_partition_raw,
    test_keep,
    "Test data",
)

all_labels = np.unique(np.concatenate([y_train, y_test]))
if len(all_labels) != 2 or 1 not in all_labels:
    raise ValueError(f"Expected binary labels containing positive class 1; found {all_labels}")

positive_label = 1
negative_label = int(all_labels[all_labels != positive_label][0])

print("\nDevelopment-set class counts")
display(class_count_table(y_train))
print("Development-set counts by partition")
display(partition_class_table(train_partition_id, y_train))
print("Final test class counts")
display(class_count_table(y_test))
print("Negative label:", negative_label)
print("Positive label:", positive_label)


## 6. Fit the QUANT transform once and cache the transformed development matrix

`QUANTTransformer` extracts quantiles from a fixed set of dyadic intervals and transformed representations of each time series. The transformation is performed once for the full development tensor, after which LOPO validation is applied to the resulting tabular features.

The supplied tensors are already standardized. Their standardization statistics were learned from the combined development split containing partitions 1, 2, 3, and 5; partition 4 remains excluded.


In [ ]:
def index_hash(indices):
    return hashlib.sha256(
        np.asarray(indices, dtype=np.int64).tobytes()
    ).hexdigest()


def array_hash(values):
    return hashlib.sha256(np.asarray(values).tobytes()).hexdigest()


def file_signature(path: Path):
    stat = path.stat()
    return {
        "name": path.name,
        "size_bytes": int(stat.st_size),
        "modified_time_ns": int(stat.st_mtime_ns),
    }


def transformed_to_numpy(matrix):
    """Convert aeon transform output to a float32 NumPy array."""
    if hasattr(matrix, "detach"):
        matrix = matrix.detach().cpu().numpy()
    return np.asarray(matrix, dtype=np.float32)


transformer_path = OUTPUT_DIR / "quant_transformer_5_features.joblib"
train_cache_path = OUTPUT_DIR / "X_train_quant_5_features.npy"
train_cache_metadata_path = OUTPUT_DIR / "X_train_quant_5_features_metadata.json"

expected_train_cache_metadata = {
    "features": FEATURES_TO_USE,
    "n_cases": int(len(y_train)),
    "case_index_hash": index_hash(train_original_indices),
    "partition_id_hash": array_hash(train_partition_id),
    "interval_depth": QUANT_INTERVAL_DEPTH,
    "quantile_divisor": QUANT_QUANTILE_DIVISOR,
    "source_x_train": file_signature(x_train_path),
    "source_y_train": file_signature(y_train_path),
    "source_partition_ids": file_signature(train_partition_path),
}

cache_is_valid = False
if (
    CACHE_TRANSFORMED_FEATURES
    and REUSE_VALID_CACHE
    and transformer_path.exists()
    and train_cache_path.exists()
    and train_cache_metadata_path.exists()
):
    with open(train_cache_metadata_path, "r") as f:
        saved_metadata = json.load(f)
    cache_is_valid = saved_metadata == expected_train_cache_metadata

if cache_is_valid:
    print("Loading the existing valid QUANT development-feature cache.")
    quant_transformer = joblib.load(transformer_path)
    X_train_quant = np.load(train_cache_path, mmap_mode="r")
else:
    print("Fitting QUANTTransformer on the development tensor...")
    quant_transformer = QUANTTransformer(
        interval_depth=QUANT_INTERVAL_DEPTH,
        quantile_divisor=QUANT_QUANTILE_DIVISOR,
    )

    start = time.time()
    transformed_train = quant_transformer.fit_transform(X_train)
    X_train_quant = transformed_to_numpy(transformed_train)
    transform_train_seconds = time.time() - start

    print(f"Development transform time: {transform_train_seconds / 60:.2f} minutes")
    print("Transformed development shape:", X_train_quant.shape)

    if CACHE_TRANSFORMED_FEATURES:
        print("Saving transformed development features and transformer...")
        np.save(train_cache_path, X_train_quant)
        joblib.dump(quant_transformer, transformer_path)
        with open(train_cache_metadata_path, "w") as f:
            json.dump(expected_train_cache_metadata, f, indent=2)

        del X_train_quant, transformed_train
        gc.collect()
        X_train_quant = np.load(train_cache_path, mmap_mode="r")

print("X_train_quant:", X_train_quant.shape, X_train_quant.dtype)


## 7. Metric, probability, and classifier helpers


In [ ]:
def positive_class_scores(
    classifier,
    X,
    positive_label=1,
    batch_size=20_000,
    description="Predicting probabilities",
):
    classes = list(classifier.classes_)
    if positive_label not in classes:
        raise ValueError(f"Positive label {positive_label} not found in {classes}")
    positive_column = classes.index(positive_label)

    score_chunks = []
    for start in tqdm(
        range(0, len(X), batch_size),
        desc=description,
        leave=False,
    ):
        stop = min(start + batch_size, len(X))
        score_chunks.append(
            classifier.predict_proba(X[start:stop])[:, positive_column]
        )
    return np.concatenate(score_chunks)


def labels_from_threshold(scores, threshold, dtype=None):
    predictions = np.where(scores >= threshold, positive_label, negative_label)
    return predictions.astype(dtype) if dtype is not None else predictions


def binary_metrics(y_true, y_pred, scores=None):
    true_positive_mask = np.asarray(y_true) == positive_label
    pred_positive_mask = np.asarray(y_pred) == positive_label

    tn, fp, fn, tp = confusion_matrix(
        true_positive_mask,
        pred_positive_mask,
        labels=[False, True],
    ).ravel()

    pod = tp / (tp + fn) if (tp + fn) else np.nan
    fpr = fp / (fp + tn) if (fp + tn) else np.nan
    far = fp / (tp + fp) if (tp + fp) else np.nan
    tss = pod - fpr if np.isfinite(pod) and np.isfinite(fpr) else np.nan

    hss_denominator = ((tp + fn) * (fn + tn)) + ((tp + fp) * (fp + tn))
    hss = (
        2 * ((tp * tn) - (fp * fn)) / hss_denominator
        if hss_denominator
        else np.nan
    )

    output = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_positive": precision_score(
            y_true, y_pred, pos_label=positive_label, zero_division=0
        ),
        "recall_positive": recall_score(
            y_true, y_pred, pos_label=positive_label, zero_division=0
        ),
        "f1_positive": f1_score(
            y_true, y_pred, pos_label=positive_label, zero_division=0
        ),
        "POD_recall": pod,
        "FPR": fpr,
        "FAR": far,
        "TSS": tss,
        "HSS": hss,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
    }

    if scores is not None:
        output["roc_auc"] = roc_auc_score(true_positive_mask, scores)
        output["average_precision_pr_auc"] = average_precision_score(
            true_positive_mask, scores
        )

    return output


def build_extra_trees(class_weight):
    return ExtraTreesClassifier(
        n_estimators=N_ESTIMATORS,
        class_weight=class_weight,
        n_jobs=-1,
        random_state=RANDOM_SEED,
    )


## 8. Run leave-one-partition-out validation

For each class weight:

- one partition is held out,
- the classifier is fitted on the other three partitions,
- probabilities are produced for the held-out partition,
- every threshold is evaluated on that partition,
- the process repeats until every development partition has served as validation once.

This produces exactly one out-of-fold probability for every development sample and class-weight candidate.


In [ ]:
fold_metric_rows = []
fold_fit_rows = []
oof_scores_by_weight = {}

for weight_name, class_weight in CLASS_WEIGHT_OPTIONS:
    print("\n" + "=" * 80)
    print("Class weight:", weight_name, class_weight)

    oof_scores = np.full(len(y_train), np.nan, dtype=np.float32)

    for held_out_partition in DEVELOPMENT_PARTITIONS:
        fold_train_idx = np.flatnonzero(train_partition_id != held_out_partition)
        fold_validation_idx = np.flatnonzero(
            train_partition_id == held_out_partition
        )

        y_fold_train = y_train[fold_train_idx]
        y_fold_validation = y_train[fold_validation_idx]

        if len(np.unique(y_fold_train)) != 2:
            raise ValueError(
                f"Fold excluding partition {held_out_partition} does not contain both classes."
            )
        if len(np.unique(y_fold_validation)) != 2:
            raise ValueError(
                f"Held-out partition {held_out_partition} does not contain both classes."
            )

        print(
            f"\nHeld-out partition {held_out_partition}: "
            f"train={len(fold_train_idx):,}, "
            f"validation={len(fold_validation_idx):,}"
        )

        classifier = build_extra_trees(class_weight)

        start = time.time()
        classifier.fit(X_train_quant[fold_train_idx], y_fold_train)
        fit_seconds = time.time() - start

        validation_scores = positive_class_scores(
            classifier,
            X_train_quant[fold_validation_idx],
            positive_label=positive_label,
            batch_size=PREDICT_BATCH_SIZE,
            description=f"Partition {held_out_partition} probabilities",
        )
        oof_scores[fold_validation_idx] = validation_scores.astype(np.float32)

        for threshold in THRESHOLDS:
            validation_pred = labels_from_threshold(
                validation_scores,
                threshold,
                dtype=y_fold_validation.dtype,
            )
            metrics = binary_metrics(
                y_fold_validation,
                validation_pred,
                scores=validation_scores,
            )
            fold_metric_rows.append({
                "class_weight_name": weight_name,
                "class_weight": str(class_weight),
                "held_out_partition": int(held_out_partition),
                "threshold": float(threshold),
                "n_fold_train": int(len(fold_train_idx)),
                "n_fold_validation": int(len(fold_validation_idx)),
                "fold_train_positive": int((y_fold_train == positive_label).sum()),
                "fold_validation_positive": int(
                    (y_fold_validation == positive_label).sum()
                ),
                "classifier_fit_seconds": float(fit_seconds),
                **metrics,
            })

        fold_fit_rows.append({
            "class_weight_name": weight_name,
            "class_weight": str(class_weight),
            "held_out_partition": int(held_out_partition),
            "n_fold_train": int(len(fold_train_idx)),
            "n_fold_validation": int(len(fold_validation_idx)),
            "classifier_fit_seconds": float(fit_seconds),
        })

        print(f"Classifier fit time: {fit_seconds / 60:.2f} minutes")
        del classifier, validation_scores
        gc.collect()

    if not np.isfinite(oof_scores).all():
        missing_count = int((~np.isfinite(oof_scores)).sum())
        raise RuntimeError(
            f"LOPO validation failed to produce {missing_count:,} out-of-fold scores "
            f"for class weight {weight_name}."
        )

    oof_scores_by_weight[weight_name] = oof_scores

fold_metrics_df = pd.DataFrame(fold_metric_rows)
fold_fit_times_df = pd.DataFrame(fold_fit_rows)

print("\nCompleted LOPO validation.")
print("Fold metric rows:", len(fold_metrics_df))
display(fold_fit_times_df)


## 9. Aggregate the LOPO results and select the winning configuration


In [ ]:
metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision_positive",
    "recall_positive",
    "f1_positive",
    "POD_recall",
    "FPR",
    "FAR",
    "TSS",
    "HSS",
    "roc_auc",
    "average_precision_pr_auc",
]

fold_summary_df = (
    fold_metrics_df
    .groupby(
        ["class_weight_name", "class_weight", "threshold"],
        as_index=False,
    )[metric_columns]
    .agg(["mean", "std", "min", "max"])
)

fold_summary_df.columns = [
    "_".join(str(part) for part in column if str(part))
    if isinstance(column, tuple)
    else str(column)
    for column in fold_summary_df.columns
]

# Normalize the names of the grouping columns after the multi-index aggregation.
rename_map = {
    "class_weight_name_": "class_weight_name",
    "class_weight_": "class_weight",
    "threshold_": "threshold",
}
fold_summary_df = fold_summary_df.rename(columns=rename_map)

pooled_rows = []
for weight_name, class_weight in CLASS_WEIGHT_OPTIONS:
    scores = oof_scores_by_weight[weight_name]
    for threshold in THRESHOLDS:
        predictions = labels_from_threshold(
            scores,
            threshold,
            dtype=y_train.dtype,
        )
        pooled_metrics = binary_metrics(y_train, predictions, scores=scores)
        pooled_rows.append({
            "class_weight_name": weight_name,
            "class_weight": str(class_weight),
            "threshold": float(threshold),
            **{f"pooled_{key}": value for key, value in pooled_metrics.items()},
        })

pooled_metrics_df = pd.DataFrame(pooled_rows)
cv_summary_df = fold_summary_df.merge(
    pooled_metrics_df,
    on=["class_weight_name", "class_weight", "threshold"],
    how="inner",
)

# Primary selection criterion: equal-weight mean TSS across held-out partitions.
# Tie-breakers: mean HSS, pooled TSS, and mean positive-class precision.
ranked_cv_df = cv_summary_df.sort_values(
    [
        "TSS_mean",
        "HSS_mean",
        "pooled_TSS",
        "precision_positive_mean",
    ],
    ascending=[False, False, False, False],
).reset_index(drop=True)

best_row = ranked_cv_df.iloc[0]
best_weight_name = best_row["class_weight_name"]
best_threshold = float(best_row["threshold"])
best_class_weight = dict(CLASS_WEIGHT_OPTIONS)[best_weight_name]

print("Winning LOPO configuration")
print("Class weight:", best_weight_name, best_class_weight)
print(f"Threshold: {best_threshold:.2f}")
print(f"Mean fold TSS: {best_row['TSS_mean']:.4f}")
print(f"TSS standard deviation: {best_row['TSS_std']:.4f}")
print(f"Mean fold HSS: {best_row['HSS_mean']:.4f}")
print(f"Pooled out-of-fold TSS: {best_row['pooled_TSS']:.4f}")

columns_to_show = [
    "class_weight_name",
    "threshold",
    "TSS_mean",
    "TSS_std",
    "TSS_min",
    "TSS_max",
    "HSS_mean",
    "balanced_accuracy_mean",
    "precision_positive_mean",
    "recall_positive_mean",
    "FPR_mean",
    "FAR_mean",
    "pooled_TSS",
    "pooled_HSS",
]
print("\nTop 15 configurations")
display(ranked_cv_df[columns_to_show].head(15))


## 10. Compare the best threshold for each class weight and inspect the winning folds


In [ ]:
best_per_weight_df = (
    cv_summary_df
    .sort_values(
        [
            "class_weight_name",
            "TSS_mean",
            "HSS_mean",
            "pooled_TSS",
            "precision_positive_mean",
        ],
        ascending=[True, False, False, False, False],
    )
    .groupby("class_weight_name", as_index=False)
    .first()
    .sort_values("TSS_mean", ascending=False)
    .reset_index(drop=True)
)

display(best_per_weight_df[columns_to_show])

winning_fold_metrics_df = (
    fold_metrics_df[
        (fold_metrics_df["class_weight_name"] == best_weight_name)
        & np.isclose(fold_metrics_df["threshold"], best_threshold)
    ]
    .sort_values("held_out_partition")
    .reset_index(drop=True)
)

winning_fold_columns = [
    "held_out_partition",
    "n_fold_train",
    "n_fold_validation",
    "fold_validation_positive",
    "threshold",
    "TSS",
    "HSS",
    "balanced_accuracy",
    "precision_positive",
    "recall_positive",
    "FPR",
    "FAR",
    "TP",
    "TN",
    "FP",
    "FN",
    "classifier_fit_seconds",
]

print("Winning configuration in each held-out partition")
display(winning_fold_metrics_df[winning_fold_columns])


## 11. Refit the winning Extra Trees classifier on all development partitions


In [ ]:
final_classifier = build_extra_trees(best_class_weight)

start = time.time()
final_classifier.fit(X_train_quant, y_train)
final_fit_seconds = time.time() - start

print("Winning class weight:", best_weight_name, best_class_weight)
print(f"Frozen threshold: {best_threshold:.2f}")
print(f"Final classifier fit time: {final_fit_seconds / 60:.2f} minutes")


## 12. Transform partition 4 and evaluate it once


In [ ]:
test_cache_path = OUTPUT_DIR / "X_test_quant_5_features.npy"
test_cache_metadata_path = OUTPUT_DIR / "X_test_quant_5_features_metadata.json"

expected_test_cache_metadata = {
    "features": FEATURES_TO_USE,
    "n_cases": int(len(y_test)),
    "case_index_hash": index_hash(test_original_indices),
    "partition_id_hash": array_hash(test_partition_id),
    "interval_depth": QUANT_INTERVAL_DEPTH,
    "quantile_divisor": QUANT_QUANTILE_DIVISOR,
    "source_x_test": file_signature(x_test_path),
    "source_y_test": file_signature(y_test_path),
    "source_partition_ids": file_signature(test_partition_path),
    "transformer_training_case_index_hash": index_hash(train_original_indices),
}

test_cache_is_valid = False
if (
    CACHE_TRANSFORMED_FEATURES
    and REUSE_VALID_CACHE
    and test_cache_path.exists()
    and test_cache_metadata_path.exists()
):
    with open(test_cache_metadata_path, "r") as f:
        saved_test_metadata = json.load(f)
    test_cache_is_valid = saved_test_metadata == expected_test_cache_metadata

if test_cache_is_valid:
    print("Loading the existing valid QUANT partition-4 feature cache.")
    X_test_quant = np.load(test_cache_path, mmap_mode="r")
else:
    print("Applying the fitted QUANT transformer to partition 4...")
    start = time.time()
    transformed_test = quant_transformer.transform(X_test)
    X_test_quant = transformed_to_numpy(transformed_test)
    test_transform_seconds = time.time() - start
    print(f"Partition-4 transform time: {test_transform_seconds / 60:.2f} minutes")

    if CACHE_TRANSFORMED_FEATURES:
        np.save(test_cache_path, X_test_quant)
        with open(test_cache_metadata_path, "w") as f:
            json.dump(expected_test_cache_metadata, f, indent=2)
        del X_test_quant, transformed_test
        gc.collect()
        X_test_quant = np.load(test_cache_path, mmap_mode="r")

print("Transformed partition-4 shape:", X_test_quant.shape)

test_scores = positive_class_scores(
    final_classifier,
    X_test_quant,
    positive_label=positive_label,
    batch_size=PREDICT_BATCH_SIZE,
    description="Partition 4 probabilities",
)

y_test_pred_default = labels_from_threshold(
    test_scores,
    DEFAULT_THRESHOLD,
    dtype=y_test.dtype,
)
y_test_pred_optimized = labels_from_threshold(
    test_scores,
    best_threshold,
    dtype=y_test.dtype,
)

metrics_default = binary_metrics(
    y_test,
    y_test_pred_default,
    scores=test_scores,
)
metrics_optimized = binary_metrics(
    y_test,
    y_test_pred_optimized,
    scores=test_scores,
)

comparison_df = pd.DataFrame([
    {
        "setting": f"winning_weight_default_threshold_{DEFAULT_THRESHOLD:.2f}",
        "class_weight_name": best_weight_name,
        "class_weight": str(best_class_weight),
        "threshold": DEFAULT_THRESHOLD,
        **metrics_default,
    },
    {
        "setting": "winning_weight_lopo_optimized_threshold",
        "class_weight_name": best_weight_name,
        "class_weight": str(best_class_weight),
        "threshold": best_threshold,
        **metrics_optimized,
    },
]).set_index("setting")

compact_columns = [
    "class_weight_name",
    "threshold",
    "accuracy",
    "balanced_accuracy",
    "precision_positive",
    "recall_positive",
    "f1_positive",
    "TSS",
    "HSS",
    "FPR",
    "FAR",
    "roc_auc",
    "average_precision_pr_auc",
    "TP",
    "TN",
    "FP",
    "FN",
]
display(comparison_df[compact_columns])

print("\nClassification report — LOPO-optimized threshold")
print(classification_report(y_test, y_test_pred_optimized, zero_division=0))

labels_for_cm = [negative_label, positive_label]
cm = confusion_matrix(y_test, y_test_pred_optimized, labels=labels_for_cm)
cm_df = pd.DataFrame(
    cm,
    index=[f"true_{value}" for value in labels_for_cm],
    columns=[f"pred_{value}" for value in labels_for_cm],
)
print("Confusion matrix — LOPO-optimized threshold")
display(cm_df)


## 13. Save validation results, predictions, transformer, and final classifier


In [ ]:
def make_json_safe(value):
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(key): make_json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_safe(item) for item in value]
    try:
        json.dumps(value)
        return value
    except TypeError:
        return str(value)


fold_metrics_path = OUTPUT_DIR / "lopo_fold_threshold_metrics.csv"
fold_fit_times_path = OUTPUT_DIR / "lopo_fold_fit_times.csv"
cv_summary_path = OUTPUT_DIR / "lopo_class_weight_threshold_summary.csv"
best_per_weight_path = OUTPUT_DIR / "lopo_best_threshold_per_class_weight.csv"
winning_fold_metrics_path = OUTPUT_DIR / "lopo_winning_configuration_by_fold.csv"
winning_oof_predictions_path = OUTPUT_DIR / "lopo_winning_oof_predictions.csv"
oof_probabilities_path = OUTPUT_DIR / "lopo_oof_probabilities_by_weight.npz"
test_comparison_path = OUTPUT_DIR / "partition4_test_metric_comparison.csv"
test_predictions_path = OUTPUT_DIR / "partition4_test_predictions.csv"
final_metrics_path = OUTPUT_DIR / "final_metrics.json"
final_classifier_path = OUTPUT_DIR / "final_extra_trees_classifier.joblib"
model_bundle_path = OUTPUT_DIR / "quant_lopo_model_bundle.joblib"

fold_metrics_df.to_csv(fold_metrics_path, index=False)
fold_fit_times_df.to_csv(fold_fit_times_path, index=False)
ranked_cv_df.to_csv(cv_summary_path, index=False)
best_per_weight_df.to_csv(best_per_weight_path, index=False)
winning_fold_metrics_df.to_csv(winning_fold_metrics_path, index=False)
comparison_df.to_csv(test_comparison_path)

winning_oof_scores = oof_scores_by_weight[best_weight_name]
winning_oof_pred = labels_from_threshold(
    winning_oof_scores,
    best_threshold,
    dtype=y_train.dtype,
)

winning_oof_predictions_df = pd.DataFrame({
    "original_training_index": train_original_indices,
    "partition_id": train_partition_id,
    "y_true": y_train,
    "positive_probability": winning_oof_scores,
    "y_pred": winning_oof_pred,
    "optimized_threshold": best_threshold,
    "winning_class_weight": best_weight_name,
})
winning_oof_predictions_df.to_csv(winning_oof_predictions_path, index=False)

np.savez_compressed(
    oof_probabilities_path,
    **{
        weight_name: scores
        for weight_name, scores in oof_scores_by_weight.items()
    },
)

test_predictions_df = pd.DataFrame({
    "original_test_index": test_original_indices,
    "partition_id": test_partition_id,
    "y_true": y_test,
    "positive_probability": test_scores,
    "y_pred_default_0_50": y_test_pred_default,
    "y_pred_lopo_optimized": y_test_pred_optimized,
    "optimized_threshold": best_threshold,
    "winning_class_weight": best_weight_name,
})
test_predictions_df.to_csv(test_predictions_path, index=False)

final_metrics = {
    "features": FEATURES_TO_USE,
    "data_dir": str(DATA_DIR),
    "development_partitions": DEVELOPMENT_PARTITIONS,
    "final_test_partition": FINAL_TEST_PARTITION,
    "validation_method": "leave-one-partition-out",
    "selection_criterion": (
        "highest mean TSS across held-out partitions; tie-break by mean HSS, "
        "pooled out-of-fold TSS, then mean positive-class precision"
    ),
    "n_development_cases": int(len(y_train)),
    "n_test_cases": int(len(y_test)),
    "development_partition_counts": {
        str(partition): int((train_partition_id == partition).sum())
        for partition in DEVELOPMENT_PARTITIONS
    },
    "class_weight_candidates": {
        name: str(value) for name, value in CLASS_WEIGHT_OPTIONS
    },
    "threshold_candidates": THRESHOLDS.tolist(),
    "winning_class_weight_name": best_weight_name,
    "winning_class_weight": best_class_weight,
    "winning_threshold": best_threshold,
    "winning_cv_summary": best_row.to_dict(),
    "winning_fold_metrics": winning_fold_metrics_df.to_dict(orient="records"),
    "partition4_test_metrics_default_threshold": metrics_default,
    "partition4_test_metrics_optimized_threshold": metrics_optimized,
    "final_classifier_fit_seconds": final_fit_seconds,
    "quant_transformer_parameters": quant_transformer.get_params(),
    "extra_trees_parameters": final_classifier.get_params(),
}

with open(final_metrics_path, "w") as f:
    json.dump(make_json_safe(final_metrics), f, indent=2)

# Save the fitted transformer even when feature caching is disabled.
joblib.dump(quant_transformer, transformer_path)

if SAVE_FINAL_MODEL:
    joblib.dump(final_classifier, final_classifier_path)
    joblib.dump(
        {
            "quant_transformer": quant_transformer,
            "classifier": final_classifier,
            "probability_threshold": best_threshold,
            "positive_label": positive_label,
            "negative_label": negative_label,
            "feature_names": FEATURES_TO_USE,
            "input_layout": "cases x channels x timepoints",
            "development_partitions": DEVELOPMENT_PARTITIONS,
            "final_test_partition": FINAL_TEST_PARTITION,
        },
        model_bundle_path,
    )

print("Saved fold threshold metrics:   ", fold_metrics_path)
print("Saved fold fit times:           ", fold_fit_times_path)
print("Saved ranked CV summary:        ", cv_summary_path)
print("Saved best threshold per weight:", best_per_weight_path)
print("Saved winning fold metrics:     ", winning_fold_metrics_path)
print("Saved winning OOF predictions:  ", winning_oof_predictions_path)
print("Saved all OOF probabilities:    ", oof_probabilities_path)
print("Saved partition-4 comparison:   ", test_comparison_path)
print("Saved partition-4 predictions:  ", test_predictions_path)
print("Saved final metrics:            ", final_metrics_path)
print("Saved QUANT transformer:        ", transformer_path)
if SAVE_FINAL_MODEL:
    print("Saved final classifier:          ", final_classifier_path)
    print("Saved model bundle:              ", model_bundle_path)


## Final result to report

Use the row labeled `winning_weight_lopo_optimized_threshold` in `partition4_test_metric_comparison.csv` as the final held-out result.

The selected class weight and probability threshold come only from leave-one-partition-out validation over partitions 1, 2, 3, and 5. Partition 4 is evaluated once after those choices are frozen.
